# Data loading phase

In [47]:
import numpy as np 
from tqdm import tqdm
import time 

# Functions

In [48]:
def unpickle(file):
    import pickle
    with open(file, 'rb') as fo:
        dict = pickle.load(fo, encoding='bytes')
    return dict
# data_batch_1=unpickle(r'C:\Users\enesm\OneDrive\Masaüstü\Computer Vision\CS231n Stanford\data\data_batch_1')

In [49]:
def get_data_points(path_of_batch_1,batch_count):
    training_data_list=[]
    for batch in range(batch_count+1):
        if batch!=0:
            base_path=path_of_batch_1[:-1]
            batch_path=base_path+f'{batch}'
            current_batch=unpickle(batch_path)
            current_batch_data=current_batch[b'data'] # first batch data here
            training_data_list.append(current_batch_data)
            training_data=np.concatenate(training_data_list,axis=0)
    return training_data

In [50]:
def get_labels(path_of_batch_1,batch_count):
    labels_list=[]
    for batch in range(batch_count+1):
        if batch!=0:
            base_path=path_of_batch_1[:-1]
            batch_path=base_path+f'{batch}'
            current_batch=unpickle(batch_path)
            current_batch_data=current_batch[b'labels'] # first batch data here
            labels_list.append(current_batch_data)
            labels=np.concatenate(labels_list,axis=0)
    return labels

# Get Train Data

In [51]:
path=r'C:\Users\enesm\OneDrive\Masaüstü\Computer Vision\CS231n Stanford\data\data_batch_1'
X_train=get_data_points(path_of_batch_1=path,batch_count=5)

In [52]:
y_train=get_labels(path_of_batch_1=path,batch_count=5)

# Get Test Data

In [53]:
X_test=get_data_points(r'C:\Users\enesm\OneDrive\Masaüstü\Computer Vision\CS231n Stanford\data\test_batch_1',1)
y_test=get_labels(path_of_batch_1=r'C:\Users\enesm\OneDrive\Masaüstü\Computer Vision\CS231n Stanford\data\test_batch_1',batch_count=1)

# Label Mappings

In [54]:
label_mappings=unpickle(r'C:\Users\enesm\OneDrive\Masaüstü\Computer Vision\CS231n Stanford\data\batches.meta')[b'label_names']
label_mappings={}
for idx, label in enumerate(label_mappings):
    label=label.decode("utf-8")
    label_mapping[idx]=label

# Linear Classifier

In [55]:
weights=np.random.random([3072,10])
weights.shape

(3072, 10)

In [144]:
class LinearClassifier:

    def __init__(self):
        self.X = None
        self.y = None
        self.W = None
        self.lambda_reg = None
        
        
    
    def forward(self):
        # Forward pass
        scores = np.dot(self.X, self.W)
        num_train = self.X.shape[0]

        # Correct labels
        correct_class_scores = scores[np.arange(num_train), self.y]

        # Loss calculation
        margins = np.maximum(0, scores - correct_class_scores[:, np.newaxis] + 1)
        margins[np.arange(num_train), self.y] = 0  # Ignore the values of correct label margins
        data_loss = np.sum(margins) / num_train
        reg_loss = np.sum(self.W**2) * self.lambda_reg / 2
        loss = data_loss + reg_loss
        
        return loss, scores, margins

    def backward(self, scores, margins):
        # Gradient of the data loss with respect to margins
        dmargins = np.zeros_like(margins)
        dmargins[margins > 0] = 1
        dmargins[np.arange(self.X.shape[0]), self.y] -= np.sum(dmargins, axis=1)

        # Gradient of the loss with respect to scores
        d_scores = dmargins / self.X.shape[0]
        
        # Gradient of the loss with respect to W
        dW = np.dot(self.X.T, d_scores)
        
        # Add the regularization gradient
        dW += self.lambda_reg * self.W

        return dW          

    def train(self, epoch, lr, X, y, W, lambda_reg):
        self.X = X
        self.y = y
        self.W = W
        self.lambda_reg = lambda_reg
        self.epoch = epoch

        for i in range(self.epoch):
            # Forward pass
            loss, scores, margins = self.forward()
            
            # Backward pass
            dW = self.backward(scores, margins)
            
            # Update weights
            self.W += lr * -dW
            
            # Optionally, compute and print loss after each epoch
            print(f"Epoch={i} the hinge loss is {loss}")

        return self.W
        
    def predict(self, X):
        # Predict function
        scores = np.dot(X, self.W)
        y_pred = np.argmax(scores, axis=1)
        return y_pred

In [145]:
classifier = LinearClassifier()
weights = np.random.random([3072,10])
classifier.train(epoch=100, lr=0.00001, X=X_train, y=y_train, W=weights, lambda_reg=0.01)


Epoch=0 the hinge loss is 9644.062497825826
Epoch=1 the hinge loss is 8498.971107062232
Epoch=2 the hinge loss is 7435.091186962904
Epoch=3 the hinge loss is 6475.603370531545
Epoch=4 the hinge loss is 5653.246406067037
Epoch=5 the hinge loss is 5001.263773147631
Epoch=6 the hinge loss is 4527.2657882335825
Epoch=7 the hinge loss is 4209.046820911775
Epoch=8 the hinge loss is 4008.9375125413667
Epoch=9 the hinge loss is 3890.154976305684
Epoch=10 the hinge loss is 3820.463941544583
Epoch=11 the hinge loss is 3777.136316164128
Epoch=12 the hinge loss is 3746.477921578534
Epoch=13 the hinge loss is 3722.11137891741
Epoch=14 the hinge loss is 3700.685128207786
Epoch=15 the hinge loss is 3680.6696593758097
Epoch=16 the hinge loss is 3661.420077076762
Epoch=17 the hinge loss is 3642.6322981514436
Epoch=18 the hinge loss is 3624.19282593242
Epoch=19 the hinge loss is 3606.050737042074
Epoch=20 the hinge loss is 3588.18202441058
Epoch=21 the hinge loss is 3570.5564646869807
Epoch=22 the hinge

array([[0.26192401, 0.17065678, 0.67934097, ..., 0.53122812, 0.52491852,
        0.66592273],
       [0.34128498, 0.82336026, 0.95407062, ..., 0.01466136, 0.11210468,
        0.83822887],
       [0.53150046, 0.75963904, 0.01311053, ..., 0.30417849, 0.98214979,
        0.81720248],
       ...,
       [0.123879  , 0.31495919, 0.82358428, ..., 0.12521501, 0.79653192,
        0.0190241 ],
       [0.16479414, 0.51163127, 0.69448976, ..., 0.89855785, 0.24920194,
        0.16837152],
       [0.77461236, 0.63768454, 0.46712464, ..., 0.0471356 , 0.44869833,
        0.39568305]])

In [146]:
predictions=classifier.predict(X_test)

In [147]:
np.mean(predictions==y_test)

0.1521

### Normal accuracy 0.2492 , duration=1796.041528224945